The following script is for use on maximum intensity projection files. It essentially quantifies total raw RFP signal intensity per macrophage (as identified by a GFP mask) as a function of distance from the center of the tumor spheroid.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skimage import io, filters, measure, morphology, transform
from scipy import ndimage as ndi
from tqdm.notebook import tqdm

def analyze_phagocytosis_raw_intensity(image_path, sample_name, pixels_per_micron=9.625, min_area=300):
    """
    Quantifies total raw RFP intensity per macrophage as a function of distance.
    """
    try:
        # 1. LOAD DATA
        img = io.imread(image_path)
        red_raw = img[0] if img.shape[0] == 2 else img[:,:,0]
        green_raw = img[1] if img.shape[0] == 2 else img[:,:,1]

        # 2. SPHEROID DETECTION (Find Center)
        ds = 4
        red_ds = transform.rescale(red_raw, 1/ds, anti_aliasing=True)
        s_mask_ds = red_ds > (filters.threshold_otsu(red_ds) * 0.9)
        labels = measure.label(s_mask_ds)
        regions = measure.regionprops(labels)
        if not regions: return None
        largest_s = max(regions, key=lambda x: x.area)
        cy, cx = np.array(largest_s.centroid) * ds

        #RFP Processing
        rfp_bg = filters.median(red_raw, morphology.disk(20))
        rfp_corrected = np.where(red_raw > rfp_bg, red_raw - rfp_bg, 0)
        
        #
        g_thresh = filters.threshold_otsu(green_raw)
        g_mask = morphology.binary_closing(green_raw > g_thresh, morphology.disk(3))
        g_mask = ndi.binary_fill_holes(g_mask)
        labeled_macs = measure.label(g_mask)

       #Intenstiy Quantification
        cell_data = []
        for prop in measure.regionprops(labeled_macs):
            if prop.area > min_area:
                min_r, min_c, max_r, max_c = prop.bbox
                
                # Extract RFP signal specifically within the green mask
                cell_rfp_crop = rfp_corrected[min_r:max_r, min_c:max_c]
                internal_values = cell_rfp_crop[prop.image]
                
                # RAW TOTAL INTENSITY
                total_raw_intensity = np.sum(internal_values)
                
                dist_px = np.sqrt((prop.centroid[0]-cy)**2 + (prop.centroid[1]-cx)**2)
                
                cell_data.append({
                    'File_Name': sample_name,
                    'Distance_um': dist_px / pixels_per_micron,
                    'Phago_Score': total_raw_intensity,
                    'Area_px': prop.area
                })
        return pd.DataFrame(cell_data)
    except Exception as e:
        print(f"Error in {sample_name}: {e}")
        return None

# --- BATCH EXECUTION ---
data_dir = '/Volumes/'  #Adjust based on file location
file_list = sorted(glob.glob(os.path.join(data_dir, "MAX_*.tif")))

all_results = []
for file_path in file_list:
    name = os.path.basename(file_path).replace(".tif", "")
    print(f"Processing: {name}") # Manual status update
    df = analyze_phagocytosis_raw_intensity(file_path, name)
    if df is not None:
        all_results.append(df)

if all_results:
    master_df = pd.concat(all_results, ignore_index=True)
    
    # Define Group based on keywords
    def get_group(name):
        if 'GFP' in name: return 'GFP'
        if '763hFC-E62K' in name: return '763hFC-E62K'
        return 'Unknown'
    
    master_df['Group'] = master_df['File_Name'].apply(get_group)
    master_df['Log_Phago_Score'] = np.log10(master_df['Phago_Score'] + 1)
    
    #Final export
    master_df.to_csv(os.path.join(data_dir, "batch_analysis_raw_intensity.csv"), index=False)
    print("\nBatch analysis complete. Raw Intensity data saved.")